# Getting started with cuPIQP

## Problem formulation

CuPIQP solves (convex) **quadratic programs (QPs)** in the following form:

$$
\begin{aligned}
\min_{x}\quad & \tfrac{1}{2}\, x^\top P x + c^\top x \\
\mathrm{s.t}\quad & A x = b, \\
                  & h_l \le G x \le h_u, \\
                  & x_l \le x \le x_u,
\end{aligned}
$$

where

| symbol | meaning | shape |
|---|---|---|
| $P = P^\top \succeq 0$ | quadratic cost (symmetric positive semidefinite) | $n \times n$ |
| $c$ | linear cost | $n$ |
| $A,\, b$ | equality constraints | $p \times n$, $\;p$ |
| $G,\, h_l,\, h_u$ | two-sided inequality constraints | $m \times n$, $\;m$, $\;m$ |
| $x_l,\, x_u$ | element-wise box bounds on $x$ | $n$, $\;n$ |

**Unbounded entries** (one-sided constraints, free variables) are set to $\pm\infty$
(use `cupy.inf`); cuPIQP detects these and drops the corresponding rows/bounds
automatically.

All problem data lives **on the GPU**: dense arrays as `cupy` arrays, or sparse
matrices as `cupyx.scipy.sparse` CSR.

cuPIQP is **natively batched** -- it solves a batch of $B$ independent QPs in a
*single* GPU call. This notebook builds a batch in which **every problem is a little
different** (including one that is **infeasible**), solves it with both the dense and
the sparse backend, and prints the per-problem solver status.

In [1]:
import cupy as cp
from cupiqp import DenseSolver, SparseSolver, Status

## Example QPs

We consider the following problem:

$$
\begin{aligned}
\min_{x_1,\,x_2}\quad & \tfrac12\bigl(6 x_1^2 + 4 x_2^2\bigr) - x_1 - 4 x_2 \\
\text{s.t.}\quad
  & x_1 - 2 x_2 = 1, \\
  & x_1 - x_2 \leq 0.2, \\
  & 2x_1 \le -1, \\
  & \theta \le x_1 \le 1,
\end{aligned}
$$

where $\theta$ is the lower bound on $x_1$ which varies among problems in the batch. In this example, we create a batch of $B=4$ problems. For simplicity, we assign different values of $\theta$ to each problem while keeping other data the same:

| problem | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| $\theta$ | $-\infty$ | $-2.0$ | $-1.0$ | $2.0$ |

Problem 3 is infeasible since the bounds of $x_1$ becomes $2 \leq x_1 \leq 1$, so
the box $2.0 \le x_1 \le 1$ is empty. 

`solver.result.x` then has shape `(B, n)` and `solver.result.info.status` is a list of
$B$ statuses (one per problem) -- so the infeasible problem reports a different status
from the rest.

In [2]:
# --- Template data ---
# quadratic + linear cost
P = cp.array([[6.0, 0.0],
              [0.0, 4.0]])
c = cp.array([-1.0, -4.0])

# equality constraint:  A x = b   (the target b is set per problem below)
A = cp.array([[1.0, -2.0]])
b = cp.array([1.0])

# two-sided inequalities:  h_l <= G x <= h_u   (use -inf / +inf for one-sided)
G   = cp.array([[1.0, -1.0],
                [2.0,  0.0]])
h_l = cp.array([-cp.inf, -cp.inf])
h_u = cp.array([0.2, -1.0])

# box bounds:  x_l <= x <= x_u
x_l = cp.array([-1.0, -cp.inf])
x_u = cp.array([ 1.0,  cp.inf])


# --- Build batched data ---
B = 4  # batch size

# replicate the shared matrices / vectors along the leading batch dimension -> (B, ...)
stack = lambda M: cp.stack([M] * B)
P_batch, c_batch, A_batch, b_batch, G_batch = stack(P), stack(c), stack(A), stack(b), stack(G)
h_l_batch, h_u_batch, x_u_batch = stack(h_l), stack(h_u), stack(x_u)
x_l_batch = cp.array([
    [-cp.inf, -cp.inf],
    [-2.0, -cp.inf],
    [-1.0, -cp.inf],
    [ 2.0, -cp.inf],  # infeasible!
])


## Dense Solver

`DenseSolver` works with **dense** cupy arrays for `P`, `A`, `G`. Hand `setup` the
batched `(B, ...)` arrays, call `solve()`, then read the per-problem solution and
status off `solver.result`.

In [3]:
solver_dense = DenseSolver()
solver_dense.settings.verbose = True

# optionally, use h_l_batch = None since h_l are all -infs
solver_dense.setup(P=P_batch, c=c_batch, A=A_batch, b=b_batch, G=G_batch,
                   h_l=h_l_batch, h_u=h_u_batch, x_l=x_l_batch, x_u=x_u_batch)
status_dense = solver_dense.solve()  # returns a list of length B containing the status of each individual problem

----------------------------------------------------------
       cuPIQP v0.1.0 - GPU-accelerated PIQP solver        
                    (c) Fenglong Song                     
   Ecole Polytechnique Federale de Lausanne (EPFL) 2026   
----------------------------------------------------------
dense backend:
batch size B = 4
variables n = 2
equality constraints p = 1
inequality constraints m = 2
inequality lower bounds n_h_l = 2
inequality upper bounds n_h_u = 2
variable lower bounds n_x_l = 2
variable upper bounds n_x_u = 2

iter  solved       gap_max     p_res_max     d_res_max     rho_max   delta_max      mu_max  p_step  d_step
   0     0/4   1.90819e+01   2.69408e+00   7.32911e+00   1.000e-06   1.000e-04   1.652e+00  0.0000  0.0000
   1     0/4   2.16919e+03   2.52134e+00   4.41792e+00   1.000e-07   1.000e-05   2.362e+01  0.0648  0.9323
   2     0/4   1.48750e+06   2.44501e+00   1.44267e+00   5.000e-08   5.000e-06   1.140e+03  0.1477  0.1575
   3     0/4   3.76528e+06   2.45375e+00

### Extract the results
The result of the optimization can be obtained from the `solver.result` object, for example:

- `solver.result.info.status`: status of the problems
- `solver.result.x`: primal solution
- `solver.result.y`: dual solution of equality constraints
- `solver.result.z_l`: dual solution of lower inequality constraints
- `solver.result.z_u`: dual solution of upper inequality constraints
- `solver.result.z_bl`: dual solution of lower bound box constraints
- `solver.result.z_bu`: dual solution of upper bound box constraints

For example,

In [4]:
# status can be obtained from solver.result.info
status_dense = solver_dense.result.info.status  # list of length B
print(f'\nresult.info.status is a {type(solver_dense.result.info.status)} object with length B:')
print(solver_dense.result.info.status)
print()


# iteration numbers
for i in range(B):
    print(f"problem {i}:  theta = {float(x_l_batch[i, 0]):.1f}   status = {solver_dense.result.info.status[i].name:<24}   iter = {solver_dense.result.info.iter[i]}")


# solutions
print(f'\nresult.info.x is a {type(solver_dense.result.x)} object with shape {solver_dense.result.x.shape}, same as (B, n)')
print(f'result.info.y is a {type(solver_dense.result.y)} object with shape {solver_dense.result.y.shape}, same as (B, p)')
print(f'result.info.z_l is a {type(solver_dense.result.z_l)} object with shape {solver_dense.result.z_l.shape}, same as (B, m) if h_l is not None in setup() otherwise (B, 0)')
print(f'result.info.z_u is a {type(solver_dense.result.z_u)} object with shape {solver_dense.result.z_u.shape}, same as (B, m) if h_u is not None in setup() otherwise (B, 0)')
print(f'result.info.z_bl is a {type(solver_dense.result.z_bl)} object with shape {solver_dense.result.z_bl.shape}, same as (B, n) if x_l is not None in setup() otherwise (B, 0)')
print(f'result.info.z_bu is a {type(solver_dense.result.z_bu)} object with shape {solver_dense.result.z_bu.shape}, same as (B, n) if x_u is not None in setup() otherwise (B, 0)')


# objectives: (B,) cupy array
print(f'\nresult.info.primal_obj is a {type(solver_dense.result.info.primal_obj)} object with shape {solver_dense.result.info.primal_obj.shape}')
print(f'result.info.dual_obj is a {type(solver_dense.result.info.dual_obj)} object with shape {solver_dense.result.info.dual_obj.shape}')


result.info.status is a <class 'list'> object with length B:
[<Status.CUPIQP_SOLVED: 0>, <Status.CUPIQP_SOLVED: 0>, <Status.CUPIQP_SOLVED: 0>, <Status.CUPIQP_PRIMAL_INFEASIBLE: 2>]

problem 0:  theta = -inf   status = CUPIQP_SOLVED              iter = 7
problem 1:  theta = -2.0   status = CUPIQP_SOLVED              iter = 7
problem 2:  theta = -1.0   status = CUPIQP_SOLVED              iter = 7
problem 3:  theta = 2.0   status = CUPIQP_PRIMAL_INFEASIBLE   iter = 18

result.info.x is a <class 'cupy.ndarray'> object with shape (4, 2), same as (B, n)
result.info.y is a <class 'cupy.ndarray'> object with shape (4, 1), same as (B, p)
result.info.z_l is a <class 'cupy.ndarray'> object with shape (4, 2), same as (B, m) if h_l is not None in setup() otherwise (B, 0)
result.info.z_u is a <class 'cupy.ndarray'> object with shape (4, 2), same as (B, m) if h_u is not None in setup() otherwise (B, 0)
result.info.z_bl is a <class 'cupy.ndarray'> object with shape (4, 2), same as (B, n) if x_l is no

To copy a cupy.ndarray to host, use [`get()`](https://docs.cupy.dev/en/stable/reference/generated/cupy.ndarray.html#cupy.ndarray.get):

In [5]:
# cupy array can be copied to host as numpy by .get(), e.g:
x_dense_host = solver_dense.result.x.get()  # (B, n) numpy array
print(f'\nresult.info.x.get() is a {type(x_dense_host)} object with shape {x_dense_host.shape}')


result.info.x.get() is a <class 'numpy.ndarray'> object with shape (4, 2)


## Sparse solver

### Prepare batched sparse matrices

`SparseSolver` expects $P, A, G$ as sparse matrices on GPU that share
the same sparsity pattern. CuPIQP accepts the following types for the `P, A, G` arguments:

- [`cupyx.scipy.sparse.csr_matrix`](https://docs.cupy.dev/en/stable/reference/generated/cupyx.scipy.sparse.csr_matrix.html#cupyx.scipy.sparse.csr_matrix). It only works for batch size 1.
- List or tuple of `cupyx.scipy.sparse.csr_matrix` objects. However, it is **discouraged** when the batch size is big because this can cause long setup time and low performance.
- `UniformBatchedCsrMatrix`. This is a class introduced and used internally by cuPIQP itself to represent a batch of sparse CSR matrices with uniform sparsity. It has `indices` and `indptr` attributes which are two `cupy.ndarray` of shape (nnz,) to represent the sparsity pattern. Its `data` attribute is a `cupy.ndarray` of shape (batch_size, nnz) that stores the non-zeros values of all matrices in the batch. We **encourage** users to import it from cuPIQP to store their batched CSR matrices and pass to the `SparseSolver`.
- [`torch.sparse_csr_tensor`](https://docs.pytorch.org/docs/2.12/generated/torch.sparse_csr_tensor.html), which is the CSR matrix representation in Torch and can express batched CSR matrices. CuPIQP requires that  `crow_indices` and `col_indices` must be identical for all matrices to enforce uniform sparsity.

The vectors $c, b, h_l, h_u, x_l, x_u$ stay stacked `(B, ...)` dense arrays just like the dense case.

In [6]:
# --- Option 1: easy, but NOT recommended for large batch
# P_sp_batch = [csr_matrix(P)] * B
# A_sp_batch = [csr_matrix(A)] * B
# G_sp_batch = [csr_matrix(G)] * B


# --- Option 2: recommended for efficiency!
from cupyx.scipy.sparse import csr_matrix
from cupiqp import UniformBatchedCsrMatrix

P_sp = csr_matrix(P)
P_sp_batch = UniformBatchedCsrMatrix(
    batch_size=B,
    indices=P_sp.indices,
    indptr=P_sp.indptr,
    data=cp.tile(P_sp.data, (B, 1))
    )

A_sp = csr_matrix(A)
A_sp_batch = UniformBatchedCsrMatrix(
    batch_size=B,
    indices=A_sp.indices,
    indptr=A_sp.indptr,
    data=cp.tile(A_sp.data, (B, 1))
    )

G_sp = csr_matrix(G)
G_sp_batch = UniformBatchedCsrMatrix(
    batch_size=B,
    indices=G_sp.indices,
    indptr=G_sp.indptr,
    data=cp.tile(G_sp.data, (B, 1))
    )

# --- alternatively, using the following:
P_sp_batch = UniformBatchedCsrMatrix.from_cupy_csr_matrix(batch_size=B, matrix=csr_matrix(P))
A_sp_batch = UniformBatchedCsrMatrix.from_cupy_csr_matrix(batch_size=B, matrix=csr_matrix(A))
G_sp_batch = UniformBatchedCsrMatrix.from_cupy_csr_matrix(batch_size=B, matrix=csr_matrix(G))

Setup solver and solve:

In [7]:
solver_sparse = SparseSolver()
solver_sparse.settings.verbose = True

solver_sparse.setup(
    P=P_sp_batch, c=c_batch,
    A=A_sp_batch, b=b_batch,
    G=G_sp_batch, h_l=h_l_batch, h_u=h_u_batch,
    x_l=x_l_batch, x_u=x_u_batch,
)

status_sparse = solver_sparse.solve()

for i in range(B):
    print(f"problem {i}:  theta = {float(x_l_batch[i, 0]):.1f}   status = {solver_sparse.result.info.status[i].name:<24}   iter = {solver_sparse.result.info.iter[i]}")

----------------------------------------------------------
       cuPIQP v0.1.0 - GPU-accelerated PIQP solver        
                    (c) Fenglong Song                     
   Ecole Polytechnique Federale de Lausanne (EPFL) 2026   
----------------------------------------------------------
sparse backend:
batch size B = 4
variables n = 2, nnz(P) = 2
equality constraints p = 1, nnz(A) = 2
inequality constraints m = 2, nnz(G) = 3
inequality lower bounds n_h_l = 2
inequality upper bounds n_h_u = 2
variable lower bounds n_x_l = 2
variable upper bounds n_x_u = 2

iter  solved       gap_max     p_res_max     d_res_max     rho_max   delta_max      mu_max  p_step  d_step
   0     0/4   1.90819e+01   2.69408e+00   7.32911e+00   1.000e-06   1.000e-04   1.652e+00  0.0000  0.0000
   1     0/4   2.16919e+03   2.52134e+00   4.41792e+00   1.000e-07   1.000e-05   2.362e+01  0.0648  0.9323
   2     0/4   1.48750e+06   2.44501e+00   1.44267e+00   5.000e-08   5.000e-06   1.140e+03  0.1477  0.1575
   

## Verify the solution consistency

In [8]:
dense_status  = [st.name for st in solver_dense.result.info.status]
sparse_status = [st.name for st in solver_sparse.result.info.status]

# the two backends should agree on every problem's status ...
assert dense_status == sparse_status

# ... and on the problems that solved, the dense and sparse optima agree.
solved = [i for i, st in enumerate(solver_dense.result.info.status)
          if st == Status.CUPIQP_SOLVED]
assert cp.allclose(
    cp.asarray(solver_dense.result.x[solved]), 
    cp.asarray(solver_sparse.result.x[solved]), 
    atol=1e-6
    )